# 01 — Equity Mean-CVaR: GPU vs CPU, honestly

This notebook reproduces the *shape* of NVIDIA's cuFOLIO `cvar_basic` problem —
~400 assets × 10,000 scenarios — as **one CVXPY model** solved by three solvers:

| solver | hardware | algorithm |
|---|---|---|
| cuOpt 26.06 PDLP | RTX 4080 (GPU) | first-order primal-dual (PDHG) |
| OR-Tools 9.15 PDLP | CPU, all threads | the *same algorithm family* on CPU |
| HiGHS 1.15 simplex | CPU, all threads | dual simplex (vertex solutions) |

The comparison discipline (see `src/bench.py` for the full protocol):
one clean subprocess per solver, discarded warmup, median of 3, hard time
limits with DNFs reported, an independent NumPy checker on every solution,
and tolerance rules disclosed rather than hidden. At this size (~4M nonzeros)
we *expect* the CPU to be competitive — the GPU crossover story is M3's
scaling study; this notebook is the harness shakedown and the first honest
calibration point against cuFOLIO's H100 marketing numbers.

In [1]:
import sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd

from src.model import build_mean_cvar
from src.scenarios import equity_scenarios

N_ASSETS, N_SCEN = 400, 10_000
R, mu = equity_scenarios(N_ASSETS, N_SCEN, seed=42)
print(f"scenario matrix: {R.shape}, nnz of CVaR block ≈ {N_SCEN * (N_ASSETS + 2):,}")

scenario matrix: (10000, 400), nnz of CVaR block ≈ 4,020,000


## The model in one look

Rockafellar–Uryasev (2000): for scenario losses $\ell_s = -(Rw)_s$,

$$\mathrm{CVaR}_\alpha(w) \;=\; \min_{t}\; t + \frac{1}{(1-\alpha)S} \sum_s \max(\ell_s - t,\, 0)$$

The $\max(\cdot,0)$ terms linearize with auxiliaries $u_s \ge 0$, $u_s \ge \ell_s - t$ —
so maximizing $\mu^\top w - \lambda\,\mathrm{CVaR}_\alpha$ under linear portfolio
constraints is **a pure LP** whose dominant block is the dense $S \times N$
scenario matrix. Dense tall blocks are exactly what GPU first-order methods eat.

In [2]:
model = build_mean_cvar(R, mu, alpha=0.95, risk_aversion=1.0, w_max=0.05)
print(f"variables: {model.n_assets + model.n_scenarios + 1:,} "
      f"(w: {model.n_assets}, u: {model.n_scenarios}, t: 1)")
print(f"CVaR tail rows: {model.n_scenarios:,}")

variables: 10,401 (w: 400, u: 10000, t: 1)
CVaR tail rows: 10,000


## One live GPU solve

A single cuOpt PDLP solve at 1e-4, validated by the independent checker
(which recomputes CVaR from the raw scenario matrix — no trust in the LP).

In [3]:
from src.bench import cuopt_pdlp_spec, run_spec

record = run_spec(model, cuopt_pdlp_spec(1e-4), n_runs=1)
print(f"status={record.status}  wall={record.wall_s_median:.3f}s  "
      f"solver-internal={record.solver_internal_s_median:.3f}s")
print(record.check.summary() if record.check else "no check")
w = model.weights()
print(f"positions >= 10bp: {(w >= 1e-3).sum()},  max weight: {w.max():.3%}")

status=optimal  wall=0.661s  solver-internal=0.581s
ok=True budget_resid=1.09e-04 bound_viol=6.94e-18 CVaR=0.013091 obj_gap=7.03e-07 dust=0
positions >= 10bp: 27,  max weight: 5.000%


## The benchmark table

Produced by `python scripts/run_bench.py --n-assets 400 --n-scenarios 10000
--tols 1e-4 1e-6 --runs 3 --crossover --out results/m1_equity_400x10k.csv`
(committed, with the full environment header). Notes for reading it:

- `wall_s_median` is end-to-end `prob.solve()`; `solver_internal_s` is the
  solver's own timer where exposed (the CVXPY OR-Tools interface discards it,
  hence the blank — its wall column is authoritative).
- `highs-simplex` runs at its **defaults** (vertex accuracy, feasibility 1e-7):
  it is the what-a-desk-uses-today baseline, *not* an equal-tolerance
  competitor to PDLP@1e-4. Tolerance-matched PDLP-vs-PDLP is the
  apples-to-apples pair.
- `gap_vs_ref_bp` is the objective distance from a HiGHS feasibility-1e-9
  reference, in basis points — the *price of loose tolerance*.
- `dust_positions` counts tiny nonzero weights (interior solutions produce
  them; crossover cleans them up — a real portfolio-hygiene concern).

In [4]:
csv_path = REPO / "results" / "m1_equity_400x10k.csv"
if csv_path.exists():
    df = pd.read_csv(csv_path, comment="#")
    cols = ["solver", "hardware", "status", "wall_s_median",
            "solver_internal_s", "gap_vs_ref_bp", "dust_positions",
            "check_ok", "n_runs"]
    display(df[cols])
    print("\nEnvironment header:")
    for line in csv_path.read_text().splitlines():
        if line.startswith("#"):
            print(" ", line)
else:
    print("results CSV not found — run scripts/run_bench.py first")

,solver,hardware,status,wall_s_median,solver_internal_s,gap_vs_ref_bp,dust_positions,check_ok,n_runs
0,cuopt-pdlp@0.0001,GPU,optimal,0.7033,0.6108,0.0033,0,True,3
1,cuopt-pdlp+xover@0.0001,GPU,optimal,1.0240,0.8020,0.0000,0,True,3
2,ortools-pdlp@0.0001,CPU,optimal,1.8209,1.6817,0.0055,0,True,3
3,cuopt-pdlp@1e-06,GPU,optimal,5.7012,5.6066,0.0001,0,True,3
4,cuopt-pdlp+xover@1e-06,GPU,optimal,5.9428,5.7120,0.0000,0,True,3
5,ortools-pdlp@1e-06,CPU,optimal,13.0846,12.9538,0.0000,0,True,3
6,highs-simplex,CPU,optimal,5.6747,5.2839,0.0000,0,True,3



Environment header:
  # timestamp_utc: 2026-07-25T07:52:32.933161+00:00
  # platform: Linux-6.6.114.1-microsoft-standard-WSL2-x86_64-with-glibc2.43
  # python: 3.12.13
  # cpu_count_logical: 24
  # cvxpy: 1.9.2
  # cuopt: 26.6.0
  # cudf: 26.6.0
  # highspy: 1.15.1
  # ortools: 9.15.6755
  # numpy: 2.4.6
  # cpu_model: 13th Gen Intel(R) Core(TM) i7-13700F
  # gpu: NVIDIA GeForce RTX 4080, 16376 MiB, 595.95
  # ram_visible_gib: 7.6
  # wsl2: yes
  # threads_cpu_solvers: ortools_pdlp=16, highs=16
  # thread_sweep_ortools_pdlp: 4T=1.965s, 8T=1.931s, 16T=1.806s, 24T=1.816s
  # thread_sweep_highs: 4T=5.345s, 8T=5.222s, 16T=5.032s, 24T=5.151s
  # time_limit_s: 300.0
  # runs_per_solver: 3
  # reference: highs-simplex feasibility 1e-9, obj=-0.0125490001
  # isolation: one clean subprocess per solver configuration
